# Semantic Kernel

在这个代码示例中，你将使用 [Semantic Kernel](https://aka.ms/ai-agents-beginners/semantic-kernel) AI 框架来创建一个基本代理。

这个示例的目标是向你展示我们稍后在实现不同代理模式的其他代码示例中将使用的步骤。

## 导入所需的 Python 包

In [ ]:
import json
import os 

from typing import Annotated

from dotenv import load_dotenv

from IPython.display import display, HTML

from openai import AsyncOpenAI

from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import FunctionCallContent, FunctionResultContent, StreamingTextContent
from semantic_kernel.functions import kernel_function

## 创建客户端

在这个示例中，我们将使用 [GitHub Models](https://aka.ms/ai-agents-beginners/github-models) 来访问 LLM。

`ai_model_id` 被定义为 `gpt-4o-mini`。尝试将模型更改为 GitHub Models 市场上可用的其他模型，以查看不同的结果。

为了使用用于 GitHub Models 的 `base_url` 的 `Azure Inference SDK`，我们将在 Semantic Kernel 中使用 `OpenAIChatCompletion` 连接器。Semantic Kernel 还提供了其他 [可用连接器](https://learn.microsoft.com/semantic-kernel/concepts/ai-services/chat-completion) 来用于其他模型提供商。

In [ ]:
import random   

# 为示例定义一个示例插件

class DestinationsPlugin:
    """随机度假目的地列表。"""

    def __init__(self):
        # 度假目的地列表
        self.destinations = [
            "Barcelona, Spain",
            "Paris, France",
            "Berlin, Germany",
            "Tokyo, Japan",
            "Sydney, Australia",
            "New York, USA",
            "Cairo, Egypt",
            "Cape Town, South Africa",
            "Rio de Janeiro, Brazil",
            "Bali, Indonesia"
        ]
        # 跟踪上次的目的地以避免重复
        self.last_destination = None

    @kernel_function(description="提供随机度假目的地。")
    def get_random_destination(self) -> Annotated[str, "返回随机度假目的地。"]:
        # 获取可用目的地（尽可能排除上次的）
        available_destinations = self.destinations.copy()
        if self.last_destination and len(available_destinations) > 1:
            available_destinations.remove(self.last_destination)

        # 选择随机目的地
        destination = random.choice(available_destinations)

        # 更新上次的目的地
        self.last_destination = destination

        return destination

In [ ]:
load_dotenv()
client = AsyncOpenAI(
    api_key=os.environ.get("GITHUB_TOKEN"), 
    base_url="https://models.inference.ai.azure.com/",
)

# 创建将被 `ChatCompletionAgent` 使用的 AI 服务
chat_completion_service = OpenAIChatCompletion(
    ai_model_id="gpt-4o-mini",
    async_client=client,
)

## 创建代理

下面我们创建一个名为 `TravelAgent` 的代理。

对于这个示例，我们使用非常简单的指令。你可以更改这些指令来查看代理的不同响应。

In [ ]:
agent = ChatCompletionAgent(
    service=chat_completion_service, 
    plugins=[DestinationsPlugin()],
    name="TravelAgent",
    instructions="你是一个有用的 AI 代理，可以帮助客户计划随机目的地的度假",
)

## 运行代理

现在我们可以通过定义 `ChatHistory` 并向其中添加 `system_message` 来运行代理。我们将使用我们之前定义的 `AGENT_INSTRUCTIONS`。

定义完这些后，我们创建一个 `user_inputs`，这将是用户发送给代理的内容。在这种情况下，我们将此消息设置为 `Plan me a sunny vacation`。

随意更改此消息以查看代理的不同响应。

In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]

async def main():
    thread: ChatHistoryAgentThread | None = None

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        agent_name = None
        full_response: list[str] = []
        function_calls: list[str] = []

        # 用于重建流式函数调用的缓冲区
        current_function_name = None
        argument_buffer = ""

        async for response in agent.invoke_stream(
            messages=user_input,
            thread=thread,
        ):
            thread = response.thread
            agent_name = response.name
            content_items = list(response.items)

            for item in content_items:
                if isinstance(item, FunctionCallContent):
                    if item.function_name:
                        current_function_name = item.function_name

                    # 累积参数（以块形式流式传输）
                    if isinstance(item.arguments, str):
                        argument_buffer += item.arguments
                elif isinstance(item, FunctionResultContent):
                    # 在显示结果之前完成任何待处理的函数调用
                    if current_function_name:
                        formatted_args = argument_buffer.strip()
                        try:
                            parsed_args = json.loads(formatted_args)
                            formatted_args = json.dumps(parsed_args)
                        except Exception:
                            pass  # 保持为原始字符串

                        function_calls.append(f"Calling function: {current_function_name}({formatted_args})")
                        current_function_name = None
                        argument_buffer = ""

                    function_calls.append(f"\nFunction Result:\n\n{item.result}")
                elif isinstance(item, StreamingTextContent) and item.text:
                    full_response.append(item.text)

        if function_calls:
            html_output += (
                "<div style='margin-bottom:10px'>"
                "<details>"
                "<summary style='cursor:pointer; font-weight:bold; color:#0066cc;'>Function Calls (click to expand)</summary>"
                "<div style='margin:10px; padding:10px; background-color:#f8f8f8; ""
                "border:1px solid #ddd; border-radius:4px; white-space:pre-wrap; font-size:14px; color:#333;'>"
                f"{chr(10).join(function_calls)}"
                "</div></details></div>"
            )

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>{agent_name or 'Assistant'}:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))

await main()